# Llama Metrics

## Joint TKG Quintuples and Triples Calculation

In [ ]:
from post_processing import get_data, geo_test_set_tkg_format, llama_preds_tkg_format

test_data = get_data(preprocessor=geo_test_set_tkg_format)
tkg_llama_preds = get_data("predictions\llama3-8B-geotkg-preds.json", preprocessor=llama_preds_tkg_format)

In [ ]:
from metrics import sample_triple_compare, sample_quintuple_compare
from sklearn.metrics import f1_score

quin_results = []
triples_results = []
stimes_res, etimes_res, events_res, subjec_res, objec_res = [], [], [], [], []  
for truth, pred in zip(test_data, tkg_llama_preds):
    quin_out, stimes, etimes, events, subjec, objec = sample_quintuple_compare(truth['quintuples'], pred['quintuples'], wrong_count=1)
    quin_results.extend(quin_out)
    stimes_res.extend(stimes)
    etimes_res.extend(etimes)
    events_res.extend(events)
    subjec_res.extend(subjec)
    objec_res.extend(objec)
    trips_out = sample_triple_compare(truth['triples'], pred['triples'])
    triples_results.extend(trips_out)

print(f"Quin: {f1_score([1]*len(quin_results), quin_results)}, triples: {f1_score([1]*len(triples_results), triples_results)}")
print(f1_score([1]*len(stimes_res), stimes_res)*100,f1_score([1]*len(etimes_res), etimes_res)*100,f1_score([1]*len(events_res), events_res)*100,f1_score([1]*len(subjec_res), subjec_res)*100,f1_score([1]*len(objec_res), objec_res)*100)

In [ ]:
llama_quins = quin_results
llama_trips = triples_results

## Isolated Normalisation

In [ ]:
from metrics import sample_norm_compare
from sklearn.metrics import f1_score
from post_processing import get_data, norm_formating

test_data = get_data("D:\\GeoTKG\\cleandata\\tie\\test.json", norm_formating)
norm_llama_preds = get_data("predictions\\llama3-8B-norm-preds.json", norm_formating)

In [ ]:
strict_norm_results = []
xxx= 0
relaxed_norm_results = []
for truth, pred in zip(test_data, norm_llama_preds):
    strict_out, relaxed_out = sample_norm_compare(truth, pred)
    strict_norm_results.extend(strict_out)
    relaxed_norm_results.extend(relaxed_out)
    xxx+=1

{"Norm relaxed":f1_score([1]*len(relaxed_norm_results), relaxed_norm_results), "Norm strict":f1_score([1]*len(strict_norm_results), strict_norm_results)}

## Isolated ET NER

In [ ]:
from metrics import sample_ner_compare, get_ner_scores
from post_processing import get_data

test_data = get_data("D:\\GeoTKG\\cleandata\\tie\\test.json")
et_ner_llama_preds = get_data("llama3-8B-et-ner-preds.json")

In [ ]:
strict_ner_et_results = []
relaxed_ner_et_results = []

for pred, truth in zip(et_ner_llama_preds, test_data):
    strict_results, relaxed_results = sample_ner_compare(truth['instances'], pred['pred'])
    strict_ner_et_results.extend(strict_results)
    relaxed_ner_et_results.extend(relaxed_results)
get_ner_scores(strict_ner_et_results, relaxed_ner_et_results)

## Isolated Geo NER

In [ ]:
from post_processing import geo_eval_formating, get_data
from metrics import sample_ner_compare, get_ner_scores

test_data = get_data("D:\\GeoTKG\\cleandata\\geo\\eval.json", preprocessor=geo_eval_formating)
geo_ner_preds = get_data("D:\\GeoTKG\\predictions\\llama3-8B-geoner-preds.json")

In [ ]:
strict_geo_results = []
relaxed_geo_results = []
for pred, truth in zip(geo_ner_preds, test_data):
    strict, relaxed = sample_ner_compare(truth, pred['pred'], geo_ner=True)
    strict_geo_results.extend(strict)
    relaxed_geo_results.extend(relaxed)
get_ner_scores(strict_geo_results, relaxed_geo_results)

# GeoTKG Metrics

In [1]:
from Pipeline import GeoTKGPipeline
import json
from post_processing import get_data, geo_test_set_tkg_format

# "D:\\GeoTKG\\cleandata\\tie\\test.json"
test_data = get_data(preprocessor=geo_test_set_tkg_format)

model = GeoTKGPipeline(model_type="pipeline", uses_ca=True)

out_preds = []
for sample in test_data:
    dcts = sample['dct']
    text = sample['text']
    output = model.pred([text], [dcts], mode="default")
    out_preds.extend(output)
    print(f"Processed {len(out_preds)}/{len(test_data)}")

Loading Pipeline with CA : True
Processed 1/8
Processed 2/8
Processed 3/8
Processed 4/8
Processed 5/8
Processed 6/8
Processed 7/8
Processed 8/8


## Joint TKG Quintuples and Triples Calculation

In [3]:
from metrics import sample_triple_compare, sample_quintuple_compare
from sklearn.metrics import f1_score

quin_results = []
triples_results = []
stimes_res, etimes_res, events_res, subjec_res, objec_res = [], [], [], [], []  
for truth, pred in zip(test_data, out_preds):
    quin_results = []
    triples_results = []
    quin_out, stimes, etimes, events, subjec, objec = sample_quintuple_compare(truth['quintuples'], pred['quintuples'], wrong_count=0)
    quin_results.extend(quin_out)
    stimes_res.extend(stimes)
    etimes_res.extend(etimes)
    events_res.extend(events)
    subjec_res.extend(subjec)
    objec_res.extend(objec)
    trips_out = sample_triple_compare(truth['triples'], pred['triples'])
    triples_results.extend(trips_out)

    print(f"Quin: {round(f1_score([1]*len(quin_results), quin_results)*100,2)}, triples: {round(f1_score([1]*len(triples_results), triples_results)*100,2)}")
print(f1_score([1]*len(stimes_res), stimes_res)*100,f1_score([1]*len(etimes_res), etimes_res)*100,f1_score([1]*len(events_res), events_res)*100,f1_score([1]*len(subjec_res), subjec_res)*100,f1_score([1]*len(objec_res), objec_res)*100)
pipeline_ca_quins = quin_results
pipeline_ca_trips = triples_results

Quin: 0.0, triples: 71.7
Quin: 25.0, triples: 38.46
Quin: 0.0, triples: 33.33
Quin: 9.09, triples: 8.7
Quin: 0.0, triples: 30.77
Quin: 0.0, triples: 27.27
Quin: 0.0, triples: 57.14
Quin: 0.0, triples: 42.11
85.56149732620321 74.85380116959064 71.08433734939759 32.8125 55.4054054054054


## Isolated ET NER

In [ ]:
from post_processing import get_data
from metrics import get_ner_scores, relaxed_correct_single, sample_ner_compare

test_data = get_data("D:\\GeoTKG\\cleandata\\tie\\test.json")

In [ ]:
strict_ner_et_results = []
relaxed_ner_et_results = []
for pred, truth in zip(out_preds, test_data):
    sorted_by_appearance = sorted(pred['events']+pred['times'], key=lambda x : x[1], reverse=False)
    strict_results, relaxed_results = sample_ner_compare(truth['instances'], pred['events']+pred['times'])
    strict_ner_et_results.extend(strict_results)
    relaxed_ner_et_results.extend(relaxed_results)
get_ner_scores(strict_ner_et_results, relaxed_ner_et_results)

## Isolated Geo NER

### Pipeline Method

In [ ]:
from models.GeoEntityModel import GeoEntityModel
import torch
from post_processing import get_data, geo_eval_formating
from transformers import AutoTokenizer
from metrics import sample_ner_compare, get_ner_scores

TOKENIZER = AutoTokenizer.from_pretrained("roberta-base", add_prefix_space=True)

GeoNER = GeoEntityModel(base="roberta-base").to(device="cuda")
load = torch.load("geotkg\\results\\geo_model\\geo_model.pt")
GeoNER.load_state_dict(load['model_state_dict'])

test_data = get_data("D:\\GeoTKG\\cleandata\\geo\\eval.json")

strict_ner_et_results = []
relaxed_ner_et_results = []
for sample in test_data:
    text = " ".join(sample["tokens"]).strip()
    (times, entities), tokens = GeoNER.predict(text, return_tokens=True)
    ids = tokens['input_ids'][0]
    instances = [[TOKENIZER.decode(ids[inst[0]:inst[1]], skip_special_tokens=True).strip(), inst[2][2:]] for inst in [*times[0],*entities[0]]]
    truth_instances = geo_eval_formating(sample)
    strict_results, relaxed_results = sample_ner_compare(truth_instances, instances, geo_ner=True)
    strict_ner_et_results.extend(strict_results)
    relaxed_ner_et_results.extend(relaxed_results)
get_ner_scores(strict_ner_et_results, relaxed_ner_et_results)



### Multi-Task Method

In [ ]:
from Pipeline import GeoTKGPipeline
import json
from post_processing import get_data

test_data = get_data("D:\\GeoTKG\\cleandata\\geo\\eval.json")

model = GeoTKGPipeline(model_type="pipeline", uses_ca=True)

out_preds = []
batch_size = 2
for i in range(0, len(test_data), batch_size):
    samples = test_data[i:i+batch_size]
    dcts = ["2020"]*len(samples)
    text = [" ".join([wrd for wrd in sample['tokens']]) for sample in samples]
    output = model.pred(text, dcts, mode="geo_isolation_testing")
    out_preds.extend(output)
    print(f"Processed {len(out_preds)}/{len(test_data)}")

In [ ]:
for i in range(len(out_preds)):
    for j in range(len(out_preds[i]['geo'])):
        out_preds[i]['geo'][j][-1] = out_preds[i]['geo'][j][-1][2:]

In [ ]:
from metrics import get_ner_scores, sample_ner_compare
from copy import deepcopy
from post_processing import geo_eval_formating

strict_ner_et_results = []
relaxed_ner_et_results = []
for sample, pred in zip(test_data, out_preds):
    truth_instances = geo_eval_formating(sample)
    strict_results, relaxed_results = sample_ner_compare(truth_instances, pred['geo'], geo_ner=True)
    strict_ner_et_results.extend(strict_results)
    relaxed_ner_et_results.extend(relaxed_results)
get_ner_scores(strict_ner_et_results, relaxed_ner_et_results)

## Isolated Normalisation
Done in training file

## Isolated ET Linking and EE Temporal Relations
Done in training file

# Sample Comaprison

In [ ]:
all_quins = 0
for sn, sample in enumerate(test_data):
    if sn == 8:
        for i in range(len(sample['quintuples'])):
            if pipeline_ca_quins[all_quins+i] == 1:
                print(sample['quintuples'][i])
    else:
        all_quins += len(sample['quintuples'])


In [11]:
out_preds[2]

{'quintuples': [{'subject': None,
   'event': 'exploration',
   'object': ['the project area'],
   's_time': datetime.datetime(1995, 1, 1, 0, 0),
   'e_time': None},
  {'subject': None,
   'event': 'focused',
   'object': ['evaluation', 'Previous exploration'],
   's_time': None,
   'e_time': None},
  {'subject': None,
   'event': 'evaluation',
   'object': ['the Ashburton and Edmund basin rocks', 'base metal deposits'],
   's_time': datetime.datetime(2017, 11, 20, 0, 0),
   'e_time': None},
  {'subject': None,
   'event': 'exploration',
   'object': None,
   's_time': datetime.datetime(1995, 1, 1, 0, 0),
   'e_time': None},
  {'subject': ['Documented previous exploration activities'],
   'event': 'commenced',
   'object': ['CRAE regional airborne geophysics', 'the early 80 s'],
   's_time': datetime.datetime(2017, 11, 20, 0, 0),
   'e_time': None},
  {'subject': ['that'],
   'event': 'generated',
   'object': ['a number'],
   's_time': datetime.datetime(2017, 11, 20, 0, 0),
   'e_time

In [ ]:
test_data[2]

In [18]:
import csv
fieldnames = ["id", "subject", "event", "object", "sTime", 'eTime']
quins = []
ents = []
for qi, quin in enumerate(out_preds[2]['quintuples']):
    if type(quin['subject'])==list:
        sub = " ".join(quin['subject'])
    else:
        sub = quin['subject']

    if type(quin['object']) == list:
        obj = " ".join(quin['object'])
    else:
        obj = quin['object']
    quins.append({"id":qi, "subject":sub, "event":quin['event'], "object":obj, "sTime":quin['s_time'], 'eTime':quin['e_time']})

with open("quins2.csv", mode='w', newline='') as file:    
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(quins)

In [16]:
with open('triples2.csv', 'w', newline='') as csvfile:
    csv_writer = csv.writer(csvfile)
    csv_writer.writerows(out_preds[2]['triples'])